# Source Detection: Stellar vs Non-Stellar

Pipeline: image → SEP background subtraction → source extraction → morphology → classify

- **Stellar** (stars): FWHM ≈ PSF, nearly circular
- **Non-stellar** (galaxies, nebulae): FWHM > PSF or elongated

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from detect import SEPDetector

## 1. Load image

In [ ]:
IMAGE_PATH = "data/test_solved.fits"

from astropy.io import fits
from astropy.wcs import WCS
import warnings
from astropy.wcs import FITSFixedWarning

with fits.open(IMAGE_PATH) as hdul:
    img = np.array(hdul[0].data / 255.0, dtype=np.float32)
    #make sure color channel is the last dimension (H, W, C) for WCS
    if img.ndim == 3 and img.shape[0] in (3, 4):
        img = np.transpose(img, (1, 2, 0))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FITSFixedWarning)
        wcs = WCS(hdul[0].header, naxis=2)  # naxis=2: always a clean 2D RA/Dec WCS
    print(f"✓ WCS loaded  (naxis={wcs.naxis})")

print(f"Shape: {img.shape}  dtype: {img.dtype}")

plt.figure(figsize=(8, 6))
plt.imshow(img,cmap='gray' if img.ndim == 2 else None)
plt.axis('off')
plt.title(IMAGE_PATH)
plt.tight_layout()
plt.show()

## 2. Detect sources

Key parameters:
- `threshold_sigma` — detection sensitivity (lower = more sources, more noise)
- `fwhm_scale_stellar` — sources with `fwhm < psf_fwhm × scale` are classified as stars
- `max_ellipticity_stellar` — stars must be rounder than this

In [ ]:
detector = SEPDetector(
    threshold_sigma=3.0,
    fwhm_scale_stellar=20,
    max_ellipticity_stellar=0.3,
)

catalog = detector.detect(img, wcs=wcs)
print(catalog)
print(f"  Stars:    {len(catalog.stars)}")
print(f"  Extended: {len(catalog.extended)}")
print(f"  PSF FWHM: {catalog.psf_fwhm:.2f} px")

In [ ]:
import matplotlib.patches as mpatches
%matplotlib inline
fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(img, cmap='gray' if img.ndim == 2 else None, origin='upper')

for src in catalog.stars:
    ax.plot(src.x, src.y, '+', color='cyan', markersize=6, markeredgewidth=0.8)

for src in catalog.extended:
    r = max(src.fwhm, 8)
    ax.add_patch(plt.Circle((src.x, src.y), r, color='orange', fill=False, linewidth=1.0))

ax.legend(handles=[
    mpatches.Patch(color='cyan',   label=f'Stellar ({len(catalog.stars)})'),
    mpatches.Patch(color='orange', label=f'Extended ({len(catalog.extended)})'),
], loc='upper right', framealpha=0.7)
ax.set_title(f"SEP detection  |  PSF FWHM = {catalog.psf_fwhm:.1f} px")
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame([
    dict(x=s.x, y=s.y, ra=s.ra, dec=s.dec,
         flux=s.flux, fwhm=s.fwhm, ellipticity=s.ellipticity, stellar=s.is_stellar)
    for s in catalog.sources
]).sort_values("flux", ascending=False)

df.head(20)

## 3. Gaia cross-match

Query Gaia DR3 for the field, then match each detected source to its nearest Gaia neighbour within `max_sep_arcsec`.

In [ ]:
from catalog import query_gaia, crossmatch

# Field info — pull from WCS or hardcode from your plate solve result
FIELD_RA    = 13.2153   # degrees
FIELD_DEC   = 56.6571   # degrees
FIELD_RADIUS = 2     # degrees — adjust to your image FOV
PIXSCALE    = 2.697     # arcsec/pixel

gaia_df = query_gaia(FIELD_RA, FIELD_DEC, radius=FIELD_RADIUS * 1.05)
print(f"Gaia sources in field: {len(gaia_df)}")
gaia_df.head(10)

In [ ]:
matches = crossmatch(catalog, gaia_df, pixscale=PIXSCALE, max_sep_arcsec=10.0)

n_matched = matches["gaia_matched"].sum()
print(f"Matched:   {n_matched} / {len(matches)} sources")
print(f"Unmatched: {len(matches) - n_matched}")

matches[matches["gaia_matched"]].sort_values("gaia_g_mag").head(20)

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(img, cmap="gray" if img.ndim == 2 else None, origin="upper")

matched   = matches[matches["gaia_matched"]]
unmatched = matches[~matches["gaia_matched"]]

ax.plot(matched["x"],   matched["y"],   "+", color="cyan",  markersize=5, markeredgewidth=0.7, label=f"Gaia matched ({len(matched)})")
ax.plot(unmatched["x"], unmatched["y"], ".", color="orange", markersize=2, alpha=0.5,           label=f"Unmatched ({len(unmatched)})")

ax.legend(loc="upper right", framealpha=0.7)
ax.set_title("Gaia cross-match")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Save catalog

In [ ]:
CACHE_PATH = "data/matches.parquet"
matches.to_parquet(CACHE_PATH, index=False)
print(f"Saved {len(matches)} sources → {CACHE_PATH}")